### Autoloader setup and streaming code

In [0]:
%run "./storage_key"

In [0]:
#Path to data for AutoLoader

storage_account_name = "konstyantyn"
container_name = "konstyantyn"
# 1000 files located
Azure_data_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/raw_lab3_1000_csv/"
# path to new files, already checked or updated files
checkpoint_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoints/lab3_autoloader/"
# path to save 
schema_location = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoints/lab3_schema/"
# target table for bronze data
target_table = "dbr_dev.konstyantyn_bronze.autoloader_stream_table"
try:     
    # start the stream
    print("Stream started")
    df_stream = (spark.readStream
        .format("cloudFiles")\
        .option("cloudFiles.format", "csv")\
        .option("header", "true")\
        .option("cloudFiles.schemaLocation", schema_location)\
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")\
        .option("cloudFiles.maxFilesPerTrigger", 50) # note for myself: this option lover use of memory and faster
        .load(Azure_data_path)
    )
    print("Stream successfully loaded data")

    # write stream to table
    query = (
        df_stream.writeStream
        .option("checkpointLocation", checkpoint_path)\
        .option("mergeSchema", "true")\
        .trigger(availableNow=True)          # this option stop stream if no new data

        .outputMode("append")\
        .toTable(target_table)
    )
    query.awaitTermination()
    print("Data successfully written to the table")
except Exception as e:
    print(e)



**Note:** The following cells display counts and sample data from the target table.

In [0]:
# Check the table
display(spark.sql(f"SELECT COUNT(*) FROM {target_table}"))
display(spark.sql(f"SELECT * FROM {target_table} LIMIT 5"))
display(spark.sql(f"SELECT * FROM {target_table} WHERE Entity = 'Poland'"))
# First Count 7884 files was there

**Note:** The following cells display counts from the target table after +1 file was added and checked.

In [0]:
#Second count test after + 1 more file added to the stream if we have 7885 files
display(spark.sql(f"SELECT COUNT(*) FROM {target_table}"))
# Second Count 7885 files now (+ 1 from the first)
